
# 01 — Methods A, B, and C: Three Strategies, Live

**Prerequisite:** Notebook `00` (the leapfrog/energy-conservation demo). If you
haven't run that, the code below will still work, but the *why* will make more
sense if you have.

**Goal:** load the actual project source code (`src/`) and watch each method
run, live, on a tiny scale, so you can see what's actually happening rather
than just reading about it.

### Recap: the three strategies

| | Weight updates | Hyperparameter updates | Correction mechanism |
|---|---|---|---|
| **Method A — Pure HHD** | HMC leapfrog | HMC leapfrog (jointly with weights) | Metropolis-Hastings accept/reject |
| **Method B — ABBO** | Adam + L-BFGS | Bayesian Optimization (outer loop) | Standard BO acquisition function |
| **Method C — Unified** | Adam → HMC → L-BFGS (3 phases) | Co-evolves with weights during HMC phase | MH correction + plateau-triggered L-BFGS |

Method A is "all physics, no safety net." Method B is "the strong practical
patchwork using existing tools." Method C fuses them: **no outer loop at
all** — the tuning happens *inside* one training run.

Let's set up the shared plumbing first.


In [ ]:

import sys, os, time

# Robust repo-path detection: this notebook assumes you've cloned/placed the
# HPO-HMC repo either (a) in the same folder as this notebook, (b) one level
# up, or (c) you set REPO_PATH manually below. Edit REPO_PATH if none of these
# match your setup.
REPO_PATH = None  # e.g. r"/path/to/HPO-HMC" -- set this if auto-detection fails
_candidates = [REPO_PATH, "HPO-HMC", os.path.join("..", "HPO-HMC"), "."]
for _c in _candidates:
    if _c and os.path.isdir(os.path.join(_c, "src")):
        REPO_PATH = _c
        break
if REPO_PATH is None:
    raise FileNotFoundError(
        "Couldn't find the HPO-HMC repo. Set REPO_PATH manually to its location.")
print(f"Using repo at: {os.path.abspath(REPO_PATH)}")

sys.path.insert(0, os.path.join(REPO_PATH, "src"))
sys.path.insert(0, REPO_PATH)

import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

from hamiltonian import HamiltonianNN, HyperparamState
from data_generator import generate_hamiltonian_data
from symplectic_solver import HamiltonianMCMC, compute_loss_and_grads
import config as base_config

# The harmonic-oscillator benchmark task: reconstruct a known energy landscape
# from (q, p) samples. Small n_samples here so this notebook runs fast.
train_loader, val_loader, _ = generate_hamiltonian_data(n_samples=500, seed=0)
criterion = nn.MSELoss()
print("Data ready. One batch looks like:")
Xb, yb = next(iter(train_loader))
print("  X (q, p) shape:", Xb.shape, " y (energy) shape:", yb.shape)



## Method A, live: pure HMC co-evolution of weights + hyperparameters

`HyperparamState` wraps the hyperparameters (learning rate, dropout, ...) as
tensors with their own momenta, exactly like the position/momentum pair from
Notebook `00`. `HamiltonianMCMC` runs the leapfrog integrator jointly over the
network's weights *and* this hyperparameter state, then applies a
Metropolis-Hastings accept/reject step (a coin-flip, weighted by how much the
joint energy changed, that decides whether to keep the proposed move).

Watch the loss and the accept/reject decisions below.


In [ ]:

hp_state = HyperparamState(base_config.INIT_HYPERPARAMS, base_config.HYPERPARAM_SPACE)
hp_state.frozen_hps = ["n_layers", "n_neurons"]  # architecture is fixed for this toy demo
hp = hp_state.decode()

model = HamiltonianNN(n_layers=hp["n_layers"], n_neurons=hp["n_neurons"],
                       dropout=hp["dropout"], input_dim=2)

# temperature=1e9 is the "optimisation mode" used throughout this project's main
# results: it effectively disables the Metropolis rejection, so HMC behaves like
# a noisy *optimiser* rather than a calibrated Bayesian sampler. We'll use a
# proper temperature=1.0 (real MCMC) here so you can actually see rejections happen.
mcmc = HamiltonianMCMC(step_size=0.005, n_leapfrog=6, mass_theta=1.0,
                        mass_lambda=base_config.MASS_LAMBDA, temperature=1.0)

Xb, yb = next(iter(train_loader))
loss, _ = compute_loss_and_grads(model, (Xb, yb), criterion)
print(f"{'step':>4} {'accepted?':>10} {'loss':>10}")
for step in range(10):
    accepted, loss = mcmc.propose(model, hp_state, (Xb, yb), criterion, loss)
    print(f"{step:>4} {str(accepted):>10} {loss:>10.4f}")

print(f"\nAcceptance rate over these 10 proposals: {mcmc.acceptance_rate:.1%}")
print("This is genuine HMC: some proposals get rejected because they would have")
print("increased the joint energy too much -- that's the physics keeping the")
print("sampler honest, exactly like the leapfrog trajectory in Notebook 00.")



## Method C, live: the three-phase curriculum

Now the actual proposed method. Three phases, back to back, in one run:

1. **Adam warmup** — fast, cheap, standard first-order descent to get roughly
   in the right neighbourhood.
2. **HMC co-evolution** — the physics-based joint exploration you just saw
   above, refining both weights and hyperparameters together.
3. **Plateau-triggered L-BFGS** — when training loss stops improving (a
   "plateau"), switch to a curvature-aware optimiser (L-BFGS) that uses
   *how the slope is changing*, not just the slope itself, to take a much
   more precise final step.

This is a **tiny, fast version** for teaching purposes (a handful of epochs
per phase) — the project's actual published numbers use much longer runs
averaged over 5 random seeds. Don't expect this toy run to match the paper's
headline MSE of 0.0033; it's here so you can see the *mechanism*, not
reproduce the full result. (Notebook `02` shows you the real, full results.)


In [ ]:

import torch.optim as optim
from copy import deepcopy

def eval_mse(model, loader, criterion):
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for Xb, yb in loader:
            total += criterion(model(Xb), yb).item()
            n += 1
    return total / max(n, 1)

torch.manual_seed(1)
hp_state = HyperparamState(base_config.INIT_HYPERPARAMS, base_config.HYPERPARAM_SPACE)
hp_state.frozen_hps = ["n_layers", "n_neurons"]
hp = hp_state.decode()
model = HamiltonianNN(n_layers=hp["n_layers"], n_neurons=hp["n_neurons"],
                       dropout=hp["dropout"], input_dim=2)

# --- Phase 1: Adam warmup ---
print("Phase 1: Adam warmup")
opt = optim.Adam(model.parameters(), lr=hp["lr"], weight_decay=1e-5)
for ep in range(5):
    model.train()
    for Xb, yb in train_loader:
        opt.zero_grad()
        criterion(model(Xb), yb).backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
val_after_warmup = eval_mse(model, val_loader, criterion)
print(f"  Validation MSE after warmup: {val_after_warmup:.4f}")

# --- Phase 2: HMC co-evolution (optimisation-mode, temperature=1e9, matching the paper) ---
print("\nPhase 2: HMC co-evolution")
mcmc = HamiltonianMCMC(step_size=0.005, n_leapfrog=6, mass_theta=1.0,
                        mass_lambda=base_config.MASS_LAMBDA, temperature=1e9)
current_loss = eval_mse(model, train_loader, criterion)
best_val, best_state = val_after_warmup, deepcopy(model.state_dict())
for ep in range(8):
    Xb, yb = next(iter(train_loader))
    accepted, current_loss = mcmc.propose(model, hp_state, (Xb, yb), criterion, current_loss)
    opt = optim.Adam(model.parameters(), lr=hp_state.decode()["lr"], weight_decay=1e-5)
    for Xb2, yb2 in train_loader:
        opt.zero_grad(); criterion(model(Xb2), yb2).backward(); opt.step()
    v = eval_mse(model, val_loader, criterion)
    if v < best_val:
        best_val, best_state = v, deepcopy(model.state_dict())
    print(f"  epoch {ep}: val MSE = {v:.4f}  (best so far: {best_val:.4f})")

# --- Phase 3: L-BFGS polish ---
print("\nPhase 3: L-BFGS curvature polish")
model.load_state_dict(best_state)
Xs, ys = [], []
for Xb, yb in train_loader:
    Xs.append(Xb); ys.append(yb)
Xf, yf = torch.cat(Xs), torch.cat(ys)
lbfgs = optim.LBFGS(model.parameters(), max_iter=20, lr=0.1, line_search_fn="strong_wolfe")
def closure():
    lbfgs.zero_grad()
    l = criterion(model(Xf), yf)
    l.backward()
    return l
lbfgs.step(closure)
final_val = eval_mse(model, val_loader, criterion)
print(f"  Final validation MSE after L-BFGS polish: {final_val:.4f}")

print(f"\nSummary: warmup={val_after_warmup:.4f} -> best-during-HMC={best_val:.4f} -> after L-BFGS={final_val:.4f}")



### What to notice

Even in this deliberately tiny run, you should typically see the validation
MSE improve at each phase transition — that's the three-phase curriculum
doing its job: fast early progress (Adam), physics-informed joint refinement
(HMC), precise final polish (L-BFGS). The ablation study (covered in Notebook
`02`) found that **L-BFGS is the single most load-bearing phase** — removing
it makes the harmonic-oscillator MSE roughly 36x worse in the full 5-seed
study.

**Next notebook (`02`):** the real, full-scale, 5-seed results for the
harmonic oscillator, plus how the project statistically tested 11 real
benchmark datasets using the Friedman and Nemenyi tests.
